# Submission 04 - Experiment 11 XGBoost

This submission uses the new best XGBoost configuration from Experiment 11.

Validation ROC-AUC: **0.941747**

The only change from the previous best model was `min_child_weight`, which was changed from 1 to 2.

The model is trained on the full training dataset and submits predicted probabilities for `Will_Buy_EV`.

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


# Find the project folder so the notebook works from VS Code.
current = Path.cwd().resolve()
project_root = None

for folder in [current, *current.parents]:
    if folder.name == "DataCompetition" and (folder / "data" / "train.csv").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the DataCompetition project folder containing data/train.csv."
    )

train_path = project_root / "data" / "train.csv"
test_path = project_root / "data" / "test.csv"
sample_path = project_root / "data" / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print(f"Project root: {project_root}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")


# Prepare the training and test data.
X = train.drop(columns=["Will_Buy_EV", "id"])
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X_test = test.drop(columns=["id"])

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")


# Experiment 11 winning configuration.
# This is the same as the previous best model except
# min_child_weight was changed from 1 to 2.
model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_processed, y)

test_predictions = model.predict_proba(X_test_processed)[:, 1]

print(f"Number of predictions: {len(test_predictions):,}")
print(f"Minimum probability: {test_predictions.min():.6f}")
print(f"Maximum probability: {test_predictions.max():.6f}")


# Create and verify the Kaggle submission.
submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})

submission_path = project_root / "submissions" / "submission_04.csv"
submission_path.parent.mkdir(parents=True, exist_ok=True)

submission.to_csv(submission_path, index=False)

assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)
assert submission["id"].equals(sample_submission["id"])
assert submission["Will_Buy_EV"].notna().all()
assert submission["Will_Buy_EV"].between(0, 1).all()

print("")
print("============================================================")
print("        SUBMISSION 04 VERIFICATION PASSED")
print("============================================================")
print("")
print(f"Saved to: {submission_path}")
print(f"Shape: {submission.shape}")
print("")
print("First 5 predictions:")
print(submission.head())
print("")
print("Model:")
print("XGBoost")
print("n_estimators = 800")
print("max_depth = 5")
print("learning_rate = 0.04")
print("min_child_weight = 2")
print("subsample = 0.85")
print("colsample_bytree = 0.85")
print("gamma = 0")
print("reg_alpha = 0")
print("reg_lambda = 1")
print("")
print("Validation ROC-AUC: 0.941747")
print("")



Project root: C:\Users\aakif\Documents\DataCompetition
Train shape: (668665, 15)
Test shape: (286571, 14)
Sample submission shape: (286571, 2)
Processed training shape: (668665, 24)
Processed test shape: (286571, 24)
Number of predictions: 286,571
Minimum probability: 0.000005
Maximum probability: 0.980439

        SUBMISSION 04 VERIFICATION PASSED

Saved to: C:\Users\aakif\Documents\DataCompetition\submissions\submission_04.csv
Shape: (286571, 2)

First 5 predictions:
       id  Will_Buy_EV
0  668665     0.010297
1  668666     0.016039
2  668667     0.005416
3  668668     0.003272
4  668669     0.021426

Model:
XGBoost
n_estimators = 800
max_depth = 5
learning_rate = 0.04
min_child_weight = 2
subsample = 0.85
colsample_bytree = 0.85
gamma = 0
reg_alpha = 0
reg_lambda = 1

Validation ROC-AUC: 0.941747



## Submission history

- Submission 01: HistGradientBoosting
- Submission 02: XGBoost baseline
- Submission 03: Tuned XGBoost T2
- Submission 04: Experiment 11 XGBoost
- Best validation ROC-AUC: **0.941747**